# Bootstrap (legacy) dataframes to S3

This notebook takes the `pandas` / `polars` / `dask` engine views from the legacy
bootstrap-migration example (`examples/data_helper_bootstrap.py`, itself a
self-contained replica of `99_Bootstrap_legacy.ipynb`'s `DataHelper` flow —
`field_map` + `sticky_filters` over a legacy-shaped table) and writes each of
them to **MinIO / S3** as Parquet, as a companion to `20_minio_s3_parquet_write.ipynb`.

| # | Topic |
|---|---|
| 1 | Setup — load credentials, configure SSRF allowlist, test connectivity |
| 2 | Bootstrap the legacy source table + `DataHelper` |
| 3 | Load the pandas / polars / dask engine views |
| 4 | Write each engine view to S3 via `ParquetSink` |
| 5 | Verify by reading each back |
| 6 | Cleanup |


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti_data").exists():
    parent = PROJECT_ROOT.parent.resolve()
    if (parent / "src" / "boti_data").exists():
        PROJECT_ROOT = parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import os
import shutil
import tempfile

from sqlalchemy import create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

from boti_data import DataHelper, ParquetReader, ParquetSink
from boti_data.connection_catalog import S3Catalog


## 1. Setup — load credentials, configure SSRF allowlist, test connectivity

Same setup as `20_minio_s3_parquet_write.ipynb`: credentials from `.env.local`, the MinIO host allowlisted for the SSRF guard, and the bucket created if it doesn't exist yet.


In [2]:
env_file = PROJECT_ROOT / ".env.local"
print(f".env.local exists: {env_file.exists()}")
print(f".env.local path: {env_file}")


.env.local exists: True
.env.local path: /Users/lvalverdeb/TeamDev/repo-split/boti-data/.env.local


In [3]:
import boti.core.filesystem as fsmod

MINIO_HOST = "10.211.55.36"
fsmod.ENDPOINT_ALLOWLIST.add(MINIO_HOST)
print(f"Allowlisted {MINIO_HOST}")
print(f"Allowlist now contains: {fsmod.ENDPOINT_ALLOWLIST}")


Allowlisted 10.211.55.36
Allowlist now contains: {'10.211.55.36'}


In [4]:
from boti.core import create_filesystem

try:
    config = fsmod.FilesystemConfig.from_env_prefix("ETL_", env_file=PROJECT_ROOT / ".env.local")
    fs = create_filesystem(config)
    try:
        items = fs.ls(config.fs_path)
    except FileNotFoundError:
        print(f"Bucket '{config.fs_path}' does not exist yet — creating it")
        fs.mkdir(config.fs_path)
        items = fs.ls(config.fs_path)
    print(f"Connected to MinIO. Bucket '{config.fs_path}' contains {len(items)} top-level items.")
    CONNECTED = True
except Exception as exc:
    print(f"Could not connect to MinIO: {exc}")
    CONNECTED = False

SCRATCH_PREFIX = f"{config.fs_path if CONNECTED else 'dst-etl'}/scratch/boti_data_bootstrap_legacy"
print(f"Scratch prefix for this notebook's demo writes: {SCRATCH_PREFIX}")


Connected to MinIO. Bucket 'dst-etl' contains 0 top-level items.
Scratch prefix for this notebook's demo writes: dst-etl/scratch/boti_data_bootstrap_legacy


## 2. Bootstrap the legacy source table + `DataHelper`

Same shape as `data_helper_bootstrap.py`: a `asm_tracking_productos`-style table, accessed through `DataHelper` with a `field_map` (legacy Spanish column names → English) and a `sticky_filters` clause that is always applied.

`worker_connection_env_var` is set so the raw DSN is never embedded directly in the helper's Dask session.


In [5]:
class Base(DeclarativeBase):
    pass


class TrackingProduct(Base):
    __tablename__ = "asm_tracking_productos"

    id: Mapped[int] = mapped_column(primary_key=True)
    id_producto: Mapped[int]
    cliente_id: Mapped[int]
    id_track_global: Mapped[int]
    id_tipo_producto: Mapped[int]


tmp_dir = tempfile.mkdtemp(prefix="boti_data_bootstrap_legacy_")
db_path = Path(tmp_dir) / "bootstrap_legacy.db"
sqlite_dsn = f"sqlite:///{db_path}"
WORKER_DSN_ENV_VAR = "BOTI_EXAMPLE_BOOTSTRAP_LEGACY_SQLITE_DSN"

engine = create_engine(sqlite_dsn)
try:
    Base.metadata.create_all(engine)
    with Session(engine) as session:
        session.add_all(
            [
                TrackingProduct(id_producto=101, cliente_id=11, id_track_global=1, id_tipo_producto=1),
                TrackingProduct(id_producto=102, cliente_id=12, id_track_global=2, id_tipo_producto=1),
                TrackingProduct(id_producto=103, cliente_id=13, id_track_global=3, id_tipo_producto=1),
                TrackingProduct(id_producto=104, cliente_id=14, id_track_global=4, id_tipo_producto=1),
                TrackingProduct(id_producto=999, cliente_id=99, id_track_global=1, id_tipo_producto=2),
            ]
        )
        session.commit()
finally:
    engine.dispose()

print(f"Local SQLite source: {db_path}")


Local SQLite source: /var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/boti_data_bootstrap_legacy_3ctmu40v/bootstrap_legacy.db


In [6]:
previous_worker_dsn = os.environ.get(WORKER_DSN_ENV_VAR)
os.environ[WORKER_DSN_ENV_VAR] = sqlite_dsn

helper = DataHelper(
    backend="sqlalchemy",
    connection_url=sqlite_dsn,
    worker_connection_env_var=WORKER_DSN_ENV_VAR,
    poolclass="sqlalchemy.pool.NullPool",
    query_only=False,
    table="asm_tracking_productos",
    field_map={
        "id_track_global": "global_track_id",
        "id_tipo_producto": "product_type_id",
    },
    sticky_filters={"product_type_id": 1},
)
columns = ["id_producto", "cliente_id", "product_type_id", "global_track_id"]
print("Bootstrap DataHelper ready (sticky_filters applies product_type_id=1)")


Bootstrap DataHelper ready (sticky_filters applies product_type_id=1)


## 3. Load the pandas / polars / dask engine views

The same `.pandas`/`.polars`/`.dask` engine-view accessors as the legacy notebook, all reading through the same `field_map`/`sticky_filters` config.


In [7]:
pandas_frame = helper.pandas.load(global_track_id__in=[1, 2, 3, 4], columns=columns)
polars_frame = helper.polars.load(global_track_id__in=[1, 2, 3, 4], columns=columns)
dask_frame = helper.dask.load(global_track_id__in=[1, 2, 3, 4], columns=columns)

print(f"pandas rows: {len(pandas_frame)}")
print(f"polars rows: {polars_frame.height}")
print(f"dask rows: {dask_frame.shape[0].compute()}")
pandas_frame.sort_values("global_track_id").reset_index(drop=True)


pandas rows: 4
polars rows: 4
dask rows: 4


,id_producto,cliente_id,product_type_id,global_track_id
0,101,11,1,1
1,102,12,1,2
2,103,13,1,3
3,104,14,1,4


## 4. Write each engine view to S3 via `ParquetSink`

`ParquetSink.write()` accepts pandas, polars, PyArrow, or Dask frames directly — no manual conversion needed. There is no date column on this legacy table, so `partition_on=None` skips partitioning (a `date_field` is only required when deriving a `partition_date` column).


In [8]:
if CONNECTED:
    s3 = S3Catalog("ETL_", env_file=PROJECT_ROOT / ".env.local")
    fs = s3.fs()

    written = {}
    for engine_name, frame in (
        ("pandas", pandas_frame),
        ("polars", polars_frame),
        ("dask", dask_frame),
    ):
        engine_path = f"{SCRATCH_PREFIX}/{engine_name}"
        with ParquetReader({"parquet_storage_path": engine_path}, fs=fs) as destination:
            with ParquetSink(destination, partition_on=None) as sink:
                result = sink.write(frame)
        written[engine_name] = result.path
        print(f"{engine_name}: wrote {len(result.files)} file(s) to {result.path}")
else:
    print("Skipping — MinIO not available")


pandas: wrote 1 file(s) to dst-etl/scratch/boti_data_bootstrap_legacy/pandas
polars: wrote 1 file(s) to dst-etl/scratch/boti_data_bootstrap_legacy/polars
dask: wrote 1 file(s) to dst-etl/scratch/boti_data_bootstrap_legacy/dask


## 5. Verify by reading each back

Read each written path back with `ParquetReader` and confirm the row counts match what was loaded from SQL.


In [9]:
if CONNECTED:
    for engine_name, path in written.items():
        with ParquetReader({"parquet_storage_path": path}, fs=fs) as reader:
            reloaded = reader.load(return_type="pandas")
        print(f"{engine_name}: reloaded {len(reloaded)} rows from {path}")
    reloaded.sort_values("global_track_id").reset_index(drop=True)
else:
    print("Skipping — MinIO not available")


pandas: reloaded 4 rows from dst-etl/scratch/boti_data_bootstrap_legacy/pandas
polars: reloaded 4 rows from dst-etl/scratch/boti_data_bootstrap_legacy/polars
dask: reloaded 4 rows from dst-etl/scratch/boti_data_bootstrap_legacy/dask


In [10]:
ttt

NameError: name 'ttt' is not defined

## 6. Cleanup

Remove everything written under this notebook's scratch prefix, close the helper, restore the worker DSN env var, and remove the local temp SQLite database.


In [ ]:
if CONNECTED:
    if fs.exists(SCRATCH_PREFIX):
        fs.rm(SCRATCH_PREFIX, recursive=True)
        print(f"Removed {SCRATCH_PREFIX} from MinIO")
    else:
        print(f"{SCRATCH_PREFIX} already absent")
else:
    print("Skipping — MinIO not available")

helper.close()
if previous_worker_dsn is None:
    os.environ.pop(WORKER_DSN_ENV_VAR, None)
else:
    os.environ[WORKER_DSN_ENV_VAR] = previous_worker_dsn

shutil.rmtree(tmp_dir, ignore_errors=True)
print("Closed helper, restored environment, removed local temp SQLite database")


### Summary

- **`data_helper_bootstrap.py`** provides the source shape reused here: a legacy-named table, `field_map` for column renaming, `sticky_filters` always applied, and pandas/polars/dask engine views off the same `DataHelper`.
- **`ParquetSink.write()`** accepts any of those frame types directly (pandas, polars, PyArrow, Dask) — no manual conversion to Dask needed before writing.
- **`partition_on=None`** skips partition-column derivation entirely for tables with no date field, instead of requiring a `date_field` that does not exist on this legacy table.
- **Cleanup** removes everything written under a `scratch/` prefix, so re-running this notebook against the shared MinIO instance does not accumulate objects.
